# Limpieza y Análisis Exploratorio de Datos
### Proyecto: Segmentación de Clientes en Centros Comerciales
**Materia:** Ciencia de Datos I  
**Institución:** ETITC  
**Autores:** Daniel Valencia, Daniel Medcalfe  
**Dataset:** Customer Shopping Dataset - Istanbul, Kaggle (2021–2023)

## 1. Importar librerías

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

## 2. Cargar el dataset

In [ ]:
df = pd.read_csv('../data/raw/customer_shopping_data.csv')

print("Filas y columnas:", df.shape)
print("\nPrimeras filas:")
df.head()

## 3. Revisión inicial

In [ ]:
print("Tipos de datos:")
print(df.dtypes)
print("\nValores nulos:")
print(df.isnull().sum())
print("\nFilas duplicadas:", df.duplicated().sum())

## 4. Limpieza de datos

In [ ]:
# Convertir fecha a tipo datetime
df['invoice_date'] = pd.to_datetime(df['invoice_date'], dayfirst=True)

# Crear columnas útiles
df['total_spend'] = df['price'] * df['quantity']
df['year']  = df['invoice_date'].dt.year
df['month'] = df['invoice_date'].dt.month

# Convertir categorías
for col in ['gender', 'category', 'payment_method', 'shopping_mall']:
    df[col] = df[col].astype('category')

print("Dataset limpio:", df.shape)
print("\nNuevos tipos:")
print(df.dtypes)

## 5. Estadísticas descriptivas

In [ ]:
df[['age', 'quantity', 'price', 'total_spend']].describe().round(2)

## 6. Distribución de edad y gasto

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['age'].hist(bins=25, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Distribución de edad')
axes[0].set_xlabel('Edad')
axes[0].set_ylabel('Frecuencia')

df['total_spend'].hist(bins=30, ax=axes[1], color='darkorange', edgecolor='white')
axes[1].set_title('Distribución del gasto por transacción')
axes[1].set_xlabel('Gasto total (USD)')
axes[1].set_ylabel('Frecuencia')

plt.tight_layout()
plt.savefig('../reports/figures/dist_edad_gasto.png')
plt.show()

## 7. Ingresos por categoría

In [ ]:
ingresos = df.groupby('category', observed=True)['total_spend'].sum().sort_values()

plt.figure(figsize=(10, 5))
ingresos.plot(kind='barh', color='steelblue')
plt.title('Ingresos totales por categoría de producto')
plt.xlabel('Ingresos (USD)')
plt.tight_layout()
plt.savefig('../reports/figures/ingresos_categoria.png')
plt.show()

print(ingresos.sort_values(ascending=False))

## 8. Distribución por género

In [ ]:
genero = df['gender'].value_counts()

plt.figure(figsize=(5, 5))
plt.pie(genero.values, labels=genero.index, autopct='%1.1f%%',
        colors=['lightcoral', 'steelblue'])
plt.title('Distribución por género')
plt.savefig('../reports/figures/genero.png')
plt.show()

## 9. Métodos de pago

In [ ]:
pago = df['payment_method'].value_counts()

plt.figure(figsize=(7, 4))
pago.plot(kind='bar', color=['steelblue', 'darkorange', 'green'], edgecolor='white')
plt.title('Métodos de pago')
plt.ylabel('Número de transacciones')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('../reports/figures/metodos_pago.png')
plt.show()

print(pago)

## 10. Ingresos por centro comercial

In [ ]:
mall = df.groupby('shopping_mall', observed=True)['total_spend'].sum().sort_values()

plt.figure(figsize=(10, 5))
mall.plot(kind='barh', color='teal')
plt.title('Ingresos por centro comercial')
plt.xlabel('Ingresos totales (USD)')
plt.tight_layout()
plt.savefig('../reports/figures/ingresos_mall.png')
plt.show()

## 11. Tendencia mensual de ingresos

In [ ]:
# Solo 2021 y 2022 (2023 tiene datos incompletos)
df_completo = df[df['year'] < 2023]

temporal = df_completo.groupby(['year', 'month'])['total_spend'].sum().reset_index()
temporal['periodo'] = pd.to_datetime(temporal['year'].astype(str) + '-' + temporal['month'].astype(str))

plt.figure(figsize=(12, 4))
plt.plot(temporal['periodo'], temporal['total_spend'], marker='o', color='steelblue', linewidth=2)
plt.fill_between(temporal['periodo'], temporal['total_spend'], alpha=0.15, color='steelblue')
plt.title('Ingresos mensuales 2021-2022')
plt.ylabel('Ingresos (USD)')
plt.xticks(rotation=35)
plt.tight_layout()
plt.savefig('../reports/figures/tendencia_temporal.png')
plt.show()

## 12. Guardar dataset limpio

In [ ]:
df.to_csv('../data/processed/datos_limpios.csv', index=False)
print("Dataset guardado en: data/processed/datos_limpios.csv")
print("Filas:", len(df))
print("Columnas:", df.columns.tolist())